In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "你是谁?"}
    ],
    stream=False,
    temperature=0.7,
)
print(completion.choices[0].message.content)

你好！我是通义千问（Qwen），阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。如果你有任何问题或需要帮助，欢迎随时告诉我！😊


In [8]:
response = client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "user", "content": "明天杭州天气如何?"}
    ],
    stream=False,
    temperature=0.9,
)
print(response.choices[0].message.content)

我无法实时获取天气信息，建议您通过以下方式查询明天（2024年X月X日）杭州的最新天气预报：

✅ 官方渠道推荐：  
- 中国气象局官网（www.cma.gov.cn）或“中国天气网”（www.weather.com.cn）→ 搜索“杭州”  
- 浙江省气象服务中心官网或“浙江天气”微信公众号  
- 手机自带天气App（如苹果天气、华为天气）或常用平台（墨迹天气、彩云天气等）

💡 小提示：杭州近期处于夏秋过渡期，多午后雷阵雨、早晚偏凉，建议关注短时强降水和气温变化，出门带伞、注意防晒与保暖。

如需，我可帮您解读天气预报中的专业术语（如“多云转阴”“相对湿度85%”“紫外线指数6”等），或提供穿衣/出行建议 😊  
请告诉我具体日期（如“10月12日”），我可以为您生成一份实用的天气应对小贴士！


# 1.加载环境变量

In [10]:
from dotenv import load_dotenv
load_dotenv()

True

# 2.定义工具

In [11]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """获取天气信息"""
    # 这里可以调用实际的天气API，这里我们模拟返回一个天气信息
    return f"{location}的天气是晴朗，温度25度。"


# 3.创建agent

In [23]:
import os
import requests
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain import hub
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
prompt = hub.pull("hwchase17/openai-tools-agent")
llm = ChatOpenAI(
    model="qwen-plus",
    api_key=os.getenv("DASHSCOPE_API_KEY"), # type: ignore
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", 
)
# ==============================================
# 🌤️ 真实天气查询工具（高德地图 API）
# ==============================================
@tool
def get_weather(city: str, date: str = "明天") -> str:
    """
    查询中国城市的实时、未来天气
    参数:
    - city: 城市名，例如 北京、上海、杭州
    - date: 日期，例如 今天、明天、后天
    """
    try:
        # 高德地图天气 API（免费申请）
        AMAP_KEY = os.getenv("GAODE_TIANQI_API_KEY")  # 等下我教你免费拿
        url = "https://restapi.amap.com/v3/weather/weatherInfo"
        
        # 1. 先获取城市编码
        city_url = "https://restapi.amap.com/v3/config/district"
        city_params = {
            "keywords": city,
            "key": AMAP_KEY,
            "subdistrict": 0,
            "output": "json"
        }
        city_resp = requests.get(city_url, params=city_params).json()
        if not city_resp.get("districts"):
            return f"未找到城市：{city}"
        city_code = city_resp["districts"][0]["adcode"]

        # 2. 获取天气
        weather_params = {
            "city": city_code,
            "key": AMAP_KEY,
            "extensions": "all",  # 获取预报
            "output": "json"
        }
        resp = requests.get(url, params=weather_params).json()
        forecasts = resp["forecasts"][0]["casts"]

        # 3. 返回对应日期天气
        if date == "今天":
            data = forecasts[0]
        elif date == "明天":
            data = forecasts[1]
        elif date == "后天":
            data = forecasts[2]
        else:
            data = forecasts[1]

        return (
            f"{date} {city} 天气：{data['dayweather']}，"
            f"温度 {data['nighttemp']}~{data['daytemp']}℃，"
            f"风向：{data['daywind']}，风力：{data['daypower']}级"
        )
    except Exception as e:
        return f"天气查询失败：{str(e)}"

tools = [get_weather]
agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
response = agent_executor.invoke({
    "input": "请帮我查询一下明天杭州的天气"
})

print("\n✅ 最终回答：", response["output"])



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': '杭州', 'date': '明天'}`


明天 杭州 天气：小雨，温度 13~19℃，风向：北，风力：1-3级明天杭州的天气为小雨，气温在13~19℃之间，风向为北，风力为1-3级。建议出门携带雨具，并注意适当添衣保暖。

> Finished chain.

✅ 最终回答： 明天杭州的天气为小雨，气温在13~19℃之间，风向为北，风力为1-3级。建议出门携带雨具，并注意适当添衣保暖。
